In [0]:
env = dbutils.widgets.get("environment") if dbutils.widgets.getAll().get("environment") else "dev"
targetTable = f"saleslake_{env}.bronze_{env}.rawDiscount"
srcPath = f"/Volumes/saleslake_{env}/bronze_{env}/vol_saleslake_src_files_{env}/daily_discount/"

spark.sql(f"""
    COPY INTO {targetTable}
    FROM (
        SELECT
            discount_id,
            discount_code,
            discount_name,
            discount_type,
            discount_value,
            NULL AS min_amount,
            max_discount_amount AS max_amount,
            valid_from,
            valid_to,
            applicable_segment,
            applicable_category,
            total_usage_limit AS usage_limit,
            is_active,
            created_date,
            current_timestamp() AS ingest_ts
        FROM '{srcPath}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header' = 'true'
    )
    COPY_OPTIONS (
        'mergeSchema' = 'false'
    )
    """)

print("Discount Bronze Load Successful")